# 🚀 Spacepresso v7 — ADL Anomaly Detection

**DINOv2 Memory Bank + Multi-layer SegHead — Ensemble 0.45/0.55**

> Leaderboard score: **0.5884**

### Architettura
- **Backbone**: DINOv2 ViT-S/14 (frozen)
- **Memory Bank**: coreset di patch features (random o greedy k-center)
- **SegHead**: testa convoluzionale multi-layer addestrata con BCE + Dice loss
- **Training data**: 30% good / 40% real anomaly / 30% cut-paste sintetico
- **Ensemble**: `W_MB * memory_bank + W_SH * seg_head`

---
**Indice**
1. Setup & Installazione
2. Mount Google Drive & Dataset
3. Configurazione
4. Moduli del progetto (`src/`)
5. Training
6. Valutazione
7. Inferenza & Submission

## 1. Setup & Installazione

In [ ]:
# Verifica GPU
import torch
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Installa dipendenze aggiuntive (scipy, sklearn, tqdm già presenti su Colab)
!pip install -q timm

## 2. Mount Google Drive & Dataset

Carica il dataset su Google Drive e aggiusta `DATA_ROOT` di conseguenza.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ⚙️ MODIFICA QUESTI PERCORSI
DATA_ROOT  = Path('/content/drive/MyDrive/ConfusionModelsADL/dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/ConfusionModelsADL/output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset root: {DATA_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Classi trovate: {[d.name for d in sorted(DATA_ROOT.iterdir()) if d.is_dir()] if DATA_ROOT.exists() else '⚠️ path non trovato'}")

## 3. Configurazione (`src/config.py`)

In [ ]:
import random
import numpy as np
import torch

# ── Iperparametri ─────────────────────────────────────────────────────────────
SEED            = 42
IMG_SIZE        = 224
PATCH_GRID      = 16
FEATURE_DIM     = 384
LAYERS_TO_USE   = [5, 8, 11]          # ViT-S/14 layers 6, 9, 12 (0-indexed)
MULTILAYER_DIM  = FEATURE_DIM * len(LAYERS_TO_USE)  # 1152

CORESET_RATIO   = 0.01   # frazione di patch mantenute nella memory bank
EPOCHS          = 50
PATIENCE        = 5      # early-stopping patience
SAMPLES_PER_EPOCH = 200
BATCH_SIZE      = 16
LR              = 1e-3
W_MB            = 0.0    # peso memory bank; peso SegHead = 1 - W_MB
BLUR_SIGMA      = 2      # sigma del Gaussian blur sulla mappa finale

# Mix training data (deve sommare a 1.0)
P_GOOD = 0.30  # batch di immagini normali
P_REAL = 0.40  # anomalie reali con GT mask
# 0.30 → cut-paste sintetico

# Percentili per normalizzazione score
P_LO = 0.5
P_HI = 99.999

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int) -> None:
    worker_seed = (torch.initial_seed() + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_all_seeds(SEED)
print("Configurazione caricata ✓")

## 4. Moduli del progetto

Le celle seguenti definiscono l'equivalente di ogni file in `src/`.

### 4.1 Modello — `SegHead` + loss (`src/model.py`)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class SegHead(nn.Module):
    """Testa di segmentazione pixel-wise su token patch concatenati di ViT.

    Input:  (B, num_patches, in_dim)
    Output: (B, 1, IMG_SIZE, IMG_SIZE) logit map
    """

    def __init__(self, in_dim: int = MULTILAYER_DIM, hidden: int = 128):
        super().__init__()
        self.conv1 = nn.Conv2d(in_dim, hidden, 1)
        self.bn1   = nn.BatchNorm2d(hidden)
        self.conv2 = nn.Conv2d(hidden, hidden, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(hidden)
        self.conv3 = nn.Conv2d(hidden, 1, 1)

    def forward(self, p: torch.Tensor) -> torch.Tensor:
        B = p.size(0)
        x = p.transpose(1, 2).reshape(B, -1, PATCH_GRID, PATCH_GRID)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x)
        return F.interpolate(x, size=(IMG_SIZE, IMG_SIZE),
                             mode='bilinear', align_corners=False)


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """BCE + soft Dice loss per anomaly segmentation."""
    bce   = F.binary_cross_entropy_with_logits(logits, target)
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice  = 1 - (2 * inter + 1.0) / (union + 1.0)
    return bce + dice.mean()


print("SegHead definito ✓")

### 4.2 Augmentation — Cut-Paste (`src/augmentation.py`)

In [ ]:
from scipy.ndimage import gaussian_filter
from PIL import Image


def get_object_mask(img: np.ndarray, threshold: int = 30) -> np.ndarray:
    return (img.mean(axis=2) if img.ndim == 3 else img) > threshold


def maybe_subcrop_large(crop_img, crop_mask, max_ratio=0.25):
    h, w = crop_mask.shape
    if (crop_mask > 0).sum() / (h * w + 1e-8) < max_ratio:
        return crop_img, crop_mask
    target_area = h * w * random.uniform(0.15, 0.30)
    sub_h = max(8, min(int(np.sqrt(target_area * h / w)), h - 2))
    sub_w = max(8, min(int(target_area / sub_h), w - 2))
    ys, xs = np.where(crop_mask > 0)
    if len(ys) == 0:
        return crop_img, crop_mask
    cy, cx = random.choice(ys), random.choice(xs)
    y0 = max(0, min(cy - sub_h // 2, h - sub_h))
    x0 = max(0, min(cx - sub_w // 2, w - sub_w))
    return (crop_img[y0:y0+sub_h, x0:x0+sub_w].copy(),
            crop_mask[y0:y0+sub_h, x0:x0+sub_w].copy())


def cut_paste(good_img, ano_img, ano_mask, scale_range=(0.4, 1.2)):
    """Incolla un patch anomalo (scalato e color-shifted) su un'immagine good."""
    H, W = good_img.shape[:2]
    ys, xs = np.where(ano_mask > 0)
    if len(ys) == 0:
        return good_img.copy(), np.zeros((H, W), dtype=np.float32)

    y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    crop_img  = ano_img[y0:y1, x0:x1].copy()
    crop_mask = ano_mask[y0:y1, x0:x1].copy()
    crop_img, crop_mask = maybe_subcrop_large(crop_img, crop_mask)
    ch, cw = crop_img.shape[:2]
    if ch < 4 or cw < 4:
        return good_img.copy(), np.zeros((H, W), dtype=np.float32)

    scale = random.uniform(*scale_range)
    nh = max(6, min(int(ch * scale), H // 2))
    nw = max(6, min(int(cw * scale), W // 2))
    crop_img  = np.array(Image.fromarray(crop_img).resize((nw, nh), Image.BILINEAR))
    crop_mask = np.array(Image.fromarray(crop_mask).resize((nw, nh), Image.NEAREST))

    shift    = np.random.randint(-15, 16, size=3).reshape(1, 1, 3)
    crop_img = np.clip(crop_img.astype(np.int16) + shift, 0, 255).astype(np.uint8)

    obj_mask = get_object_mask(good_img)
    oys, oxs = np.where(obj_mask)
    if len(oys) == 0:
        py = random.randint(0, H - nh)
        px = random.randint(0, W - nw)
    else:
        oy0, oy1 = oys.min(), oys.max()
        ox0, ox1 = oxs.min(), oxs.max()
        py = random.randint(max(0, oy0 - nh//4), max(0, min(H-nh, oy1 - nh//2)))
        px = random.randint(max(0, ox0 - nw//4), max(0, min(W-nw, ox1 - nw//2)))

    synth    = good_img.copy()
    out_mask = np.zeros((H, W), dtype=np.float32)
    alpha    = gaussian_filter((crop_mask > 0).astype(np.float32), sigma=1.0)
    region   = synth[py:py+nh, px:px+nw]
    synth[py:py+nh, px:px+nw] = (
        region * (1 - alpha[:, :, None]) + crop_img * alpha[:, :, None]
    ).astype(np.uint8)
    out_mask[py:py+nh, px:px+nw] = alpha
    return synth, out_mask


print("Augmentation definita ✓")

### 4.3 Dataset (`src/datasets.py`)

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class ImageFolderDataset(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = paths, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB'))


class TestDataset(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = paths, tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        path = self.paths[i]
        return self.tf(Image.open(path).convert('RGB')), Path(path).name


def _resize_mask(mask: np.ndarray) -> np.ndarray:
    if mask.shape == (IMG_SIZE, IMG_SIZE):
        return (mask > 0).astype(np.float32)
    pil = Image.fromarray((mask > 0).astype(np.uint8) * 255)
    pil = pil.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    return (np.array(pil) > 127).astype(np.float32)


class TrainSynthDataset(Dataset):
    """30% good / 40% real anomaly / 30% cut-paste sintetico."""

    def __init__(self, good_paths, sources, n, tf,
                 p_good=P_GOOD, p_real=P_REAL):
        self.good_paths = good_paths
        self.sources    = sources
        self.n, self.tf = n, tf
        self.p_good, self.p_real = p_good, p_real

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        r = random.random()
        if r < self.p_good or not self.sources:
            img = np.array(Image.open(random.choice(self.good_paths)).convert('RGB'))
            return self.tf(Image.fromarray(img)), torch.zeros(IMG_SIZE, IMG_SIZE)
        if r < self.p_good + self.p_real:
            src  = random.choice(self.sources)
            mask = _resize_mask(src['mask'])
            return self.tf(Image.fromarray(src['image'])), torch.from_numpy(mask).float()
        good_img = np.array(Image.open(random.choice(self.good_paths)).convert('RGB'))
        src = random.choice(self.sources)
        synth, mask = cut_paste(good_img, src['image'], src['mask'])
        return self.tf(Image.fromarray(synth)), torch.from_numpy(mask).float()


print("Dataset definiti ✓")

### 4.4 Estrazione features (`src/features.py`)

In [ ]:
from torch.utils.data import DataLoader


@torch.no_grad()
def extract_multilayer_patches(model, x, layers=LAYERS_TO_USE):
    """Estrae e concatena i patch token da più layer ViT.
    Output: (B, num_patches, FEATURE_DIM * len(layers))
    """
    intermediates = model.get_intermediate_layers(
        x, n=layers, reshape=False, return_class_token=False, norm=True
    )
    return torch.cat(intermediates, dim=2)


@torch.no_grad()
def extract_patch_features(model, paths, batch_size=32):
    """Estrae patch token dell'ultimo layer per una lista di path.
    Output: (N * num_patches, FEATURE_DIM) CPU float32
    """
    g = torch.Generator()
    g.manual_seed(0)
    loader = DataLoader(
        ImageFolderDataset(paths, preprocess),
        batch_size=batch_size, num_workers=2,
        pin_memory=True, generator=g, worker_init_fn=worker_init_fn,
    )
    feats = []
    for batch in loader:
        out = model.forward_features(batch.to(device, non_blocking=True))
        feats.append(out['x_norm_patchtokens'].reshape(-1, FEATURE_DIM).cpu())
    return torch.cat(feats, dim=0)


print("Feature extraction definita ✓")

### 4.5 Coreset (`src/coreset.py`)

In [ ]:
@torch.no_grad()
def greedy_coreset(features, n_samples, seed=SEED):
    """k-center greedy coreset selection (O(n * n_samples))."""
    n = len(features)
    n_samples = min(n_samples, n)
    if n_samples >= n:
        return features
    feats_gpu = features.to(device)
    rng = torch.Generator()
    rng.manual_seed(seed)
    start    = int(torch.randint(0, n, (1,), generator=rng))
    selected = [start]
    min_dist = torch.cdist(feats_gpu, feats_gpu[start:start+1]).squeeze(1)
    for _ in range(n_samples - 1):
        idx = int(min_dist.argmax())
        selected.append(idx)
        d = torch.cdist(feats_gpu, feats_gpu[idx:idx+1]).squeeze(1)
        torch.minimum(min_dist, d, out=min_dist)
    return features[torch.tensor(selected, dtype=torch.long)]


def random_subsample(features, n_samples, seed=SEED):
    """Sottocampionamento random O(N)."""
    n = len(features)
    n_samples = min(n_samples, n)
    rng = np.random.RandomState(seed)
    idx = rng.choice(n, size=n_samples, replace=False)
    return features[torch.from_numpy(idx).long()]


def coreset_subsample(features, ratio=CORESET_RATIO, seed=SEED, method='random'):
    """Sottocampiona una frazione di patch features per la memory bank."""
    n_samples = max(1, int(len(features) * ratio))
    if method == 'greedy':
        return greedy_coreset(features, n_samples, seed=seed)
    return random_subsample(features, n_samples, seed=seed)


print("Coreset definito ✓")

### 4.6 Checkpoint (`src/checkpoint.py`)

In [ ]:
import json
from datetime import datetime


def save_run(output_dir, memory_banks, seg_heads,
             config_dict, metrics=None):
    """Salva memoria, pesi e metriche in una cartella timestampata."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = output_dir / 'runs' / timestamp
    (run_dir / 'memory_banks').mkdir(parents=True, exist_ok=True)
    (run_dir / 'seg_heads').mkdir(parents=True, exist_ok=True)

    for cls, bank in memory_banks.items():
        torch.save(bank, run_dir / 'memory_banks' / f'{cls}.pt')
    for cls, head in seg_heads.items():
        torch.save(head.state_dict(), run_dir / 'seg_heads' / f'{cls}.pt')

    with open(run_dir / 'config.json', 'w') as f:
        json.dump(config_dict, f, indent=2)

    if metrics is not None:
        eval_dir = run_dir / 'evaluation'
        eval_dir.mkdir(exist_ok=True)
        with open(eval_dir / 'metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)

    print(f'\nRun salvato → {run_dir}')
    return run_dir


def load_run(run_dir):
    """Ripristina memory banks e SegHead da una run salvata."""
    memory_banks = {}
    for pt in sorted((run_dir / 'memory_banks').glob('*.pt')):
        memory_banks[pt.stem] = torch.load(pt, map_location=device)
    seg_heads = {}
    for pt in sorted((run_dir / 'seg_heads').glob('*.pt')):
        head = SegHead()
        head.load_state_dict(torch.load(pt, map_location=device))
        seg_heads[pt.stem] = head.to(device).eval()
    return memory_banks, seg_heads


print("Checkpoint definito ✓")

### 4.7 Inferenza (`src/inference.py`)

In [ ]:
from tqdm import tqdm

_TTA_FLIPS = [[], [-1]]  # originale + h-flip


@torch.no_grad()
def memorybank_infer(model, paths, bank, batch_size=32, seed=SEED):
    """Score basato su distanza coseno dal nearest neighbour nella memory bank."""
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(
        TestDataset(paths, preprocess), batch_size=batch_size,
        num_workers=2, pin_memory=True, generator=g,
        worker_init_fn=worker_init_fn,
    )
    bank_gpu = bank.to(device)
    scores, fns = [], []
    for imgs, names in tqdm(loader, desc='MB infer', leave=False):
        imgs    = imgs.to(device, non_blocking=True)
        out     = model.forward_features(imgs)
        patches = F.normalize(out['x_norm_patchtokens'], p=2, dim=2)
        max_sim, _ = (patches @ bank_gpu.T).max(dim=2)
        dist = (1.0 - max_sim).reshape(-1, PATCH_GRID, PATCH_GRID)
        s = F.interpolate(
            dist.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE),
            mode='bilinear', align_corners=False,
        ).squeeze(1)
        scores.append(s.cpu().numpy())
        fns.extend(names)
    return np.concatenate(scores, axis=0), fns


@torch.no_grad()
def seghead_infer(model, paths, head, batch_size=32, seed=SEED, tta=True):
    """Score con SegHead multi-layer (+ TTA h-flip opzionale)."""
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(
        TestDataset(paths, preprocess), batch_size=batch_size,
        num_workers=2, pin_memory=True, generator=g,
        worker_init_fn=worker_init_fn,
    )
    head.eval()
    scores, fns = [], []
    for imgs, names in tqdm(loader, desc='SH infer', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        aug_scores = []
        for flip_dims in (_TTA_FLIPS if tta else [[]]):
            x       = torch.flip(imgs, flip_dims) if flip_dims else imgs
            patches = extract_multilayer_patches(model, x)
            s       = torch.sigmoid(head(patches)).squeeze(1)
            if flip_dims:
                s = torch.flip(s, flip_dims)
            aug_scores.append(s)
        s = torch.stack(aug_scores).mean(0).cpu().numpy()
        scores.append(s)
        fns.extend(names)
    return np.concatenate(scores, axis=0), fns


def run_inference(model, data_root, memory_banks, seg_heads):
    """Inferenza su split good-train e test per tutte le classi."""
    print('\nRunning inference...')
    mb_train, mb_test, sh_train, sh_test = {}, {}, {}, {}
    for class_name in sorted(seg_heads.keys()):
        good = [str(p) for p in sorted((data_root / class_name / 'train' / 'good').glob('*.png'))]
        test = [str(p) for p in sorted((data_root / class_name / 'test').glob('*.png'))]
        if W_MB > 0:
            mb_tr, _    = memorybank_infer(model, good, memory_banks[class_name])
            mb_te, mfns = memorybank_infer(model, test, memory_banks[class_name])
            mb_train[class_name] = mb_tr
            mb_test[class_name]  = dict(zip(mfns, mb_te))
        sh_tr, _    = seghead_infer(model, good, seg_heads[class_name])
        sh_te, sfns = seghead_infer(model, test, seg_heads[class_name])
        sh_train[class_name] = sh_tr
        sh_test[class_name]  = dict(zip(sfns, sh_te))
        print(f'  {class_name} done')
    return mb_train, mb_test, sh_train, sh_test


print("Inferenza definita ✓")

### 4.8 Training (`src/training.py`)

In [ ]:
import copy
import gc
from collections import defaultdict
from sklearn.metrics import average_precision_score


def build_memory_banks(model, data_root, coreset_ratio=CORESET_RATIO,
                       seed=SEED, coreset_method='random'):
    """Costruisce memory bank L2-normalizzate per ogni classe."""
    print(f'Building memory banks (method={coreset_method}, ratio={coreset_ratio*100:.1f}%)...')
    banks = {}
    for class_dir in sorted(data_root.iterdir()):
        good_dir = class_dir / 'train' / 'good'
        if not good_dir.is_dir():
            continue
        class_name = class_dir.name
        paths = [str(p) for p in sorted(good_dir.glob('*.png'))]
        feats = extract_patch_features(model, paths)
        bank  = F.normalize(
            coreset_subsample(feats, ratio=coreset_ratio,
                              seed=seed, method=coreset_method),
            p=2, dim=1,
        )
        banks[class_name] = bank
        print(f'  {class_name}: {len(paths)} imgs → bank {tuple(bank.shape)}')
        del feats
        gc.collect()
    return banks


def collect_anomaly_sources(data_root):
    """Carica coppie (immagine, mask) e splitta 3/2 tra train e val."""
    train_sources = defaultdict(list)
    val_items     = defaultdict(list)
    for class_dir in sorted(data_root.iterdir()):
        train_dir = class_dir / 'train'
        gt_dir    = class_dir / 'ground_truth_train'
        if not (train_dir.is_dir() and gt_dir.is_dir()):
            continue
        class_name = class_dir.name
        for ano_gt_dir in sorted(gt_dir.iterdir()):
            ano_img_dir = train_dir / ano_gt_dir.name
            pairs = []
            for mask_path in sorted(ano_gt_dir.glob('*.png')):
                mask = np.array(Image.open(mask_path).convert('L'))
                if mask.max() == 0:
                    continue
                pairs.append((ano_img_dir / mask_path.name, mask))
            for img_path, mask in pairs[:3]:
                img = np.array(Image.open(img_path).convert('RGB'))
                train_sources[class_name].append({'image': img, 'mask': mask})
            for img_path, mask in pairs[3:]:
                val_items[class_name].append({'path': str(img_path), 'mask': mask})
    return train_sources, val_items


@torch.no_grad()
def _val_pixel_ap(model, head, val_items):
    head.eval()
    all_preds, all_gt = [], []
    for item in val_items:
        img     = preprocess(Image.open(item['path']).convert('RGB')).unsqueeze(0).to(device)
        patches = extract_multilayer_patches(model, img)
        score   = torch.sigmoid(head(patches)).squeeze().cpu().numpy()
        mask    = (item['mask'] > 0).astype(np.uint8)
        if mask.shape != (IMG_SIZE, IMG_SIZE):
            mask = (np.array(
                Image.fromarray(mask * 255).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            ) > 127).astype(np.uint8)
        all_preds.append(score.flatten())
        all_gt.append(mask.flatten())
    gt = np.concatenate(all_gt)
    if gt.max() == 0:
        return 0.0
    return float(average_precision_score(gt, np.concatenate(all_preds)))


def train_seg_heads(model, data_root, anomaly_sources, val_sources,
                    epochs=EPOCHS, patience=PATIENCE, seed=SEED):
    """Addestra un SegHead per classe con early stopping su val AP."""
    print('\nTraining multi-layer SegHeads...')
    seg_heads = {}
    for cls_idx, class_name in enumerate(sorted(anomaly_sources.keys())):
        good_paths = [str(p) for p in sorted(
            (data_root / class_name / 'train' / 'good').glob('*.png')
        )]
        sources  = anomaly_sources[class_name]
        cls_seed = seed + cls_idx
        g = torch.Generator()
        g.manual_seed(cls_seed)

        loader = DataLoader(
            TrainSynthDataset(good_paths, sources, SAMPLES_PER_EPOCH, preprocess),
            batch_size=BATCH_SIZE, num_workers=2, pin_memory=True,
            generator=g, worker_init_fn=worker_init_fn,
        )
        head  = SegHead().to(device)
        optim = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)

        class_val  = val_sources.get(class_name, [])
        best_ap    = -1.0
        best_state = None
        no_improve = 0

        for epoch in range(epochs):
            head.train()
            epoch_loss = 0.0
            for imgs, masks in loader:
                imgs  = imgs.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).unsqueeze(1)
                with torch.no_grad():
                    patches = extract_multilayer_patches(model, imgs)
                loss = bce_dice_loss(head(patches), masks)
                optim.zero_grad()
                loss.backward()
                optim.step()
                epoch_loss += loss.item()

            if class_val:
                val_ap = _val_pixel_ap(model, head, class_val)
                head.train()
                if val_ap > best_ap:
                    best_ap    = val_ap
                    best_state = copy.deepcopy(head.state_dict())
                    no_improve = 0
                else:
                    no_improve += 1

            if (epoch + 1) % 10 == 0 or no_improve == patience:
                ap_str = f'  val_ap={best_ap:.4f}  no_improve={no_improve}/{patience}' if class_val else ''
                print(f'  {class_name} epoch {epoch+1:2d}/{epochs}  '
                      f'loss={epoch_loss/len(loader):.4f}{ap_str}')

            if no_improve >= patience:
                print(f'  {class_name} early stop at epoch {epoch+1}')
                break

        if best_state is not None:
            head.load_state_dict(best_state)
        head.eval()
        seg_heads[class_name] = head
        msg = f'  {class_name} done  (best val_ap={best_ap:.4f})' if class_val else f'  {class_name} done'
        print(msg)
    return seg_heads


print("Training definito ✓")

### 4.9 Valutazione (`src/evaluation.py`)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve


def _load_good_paths(data_root, class_name, max_good=150, seed=0):
    all_good = sorted((data_root / class_name / 'train' / 'good').glob('*.png'))
    if max_good is not None and len(all_good) > max_good:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(all_good), size=max_good, replace=False)
        all_good = [all_good[i] for i in sorted(idx)]
    return [str(p) for p in all_good]


def _resize_mask_eval(mask):
    return (np.array(
        Image.fromarray(mask).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    ) > 0).astype(np.uint8)


def _save_heatmaps(items, ens_scores, class_name, save_path, n=4):
    n = min(n, len(items))
    if n == 0:
        return
    order = np.argsort([s.max() for s in ens_scores])[::-1][:n]
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5*n), squeeze=False)
    for col, title in enumerate(['Original', 'GT Mask', 'Score Map', 'Overlay']):
        axes[0, col].set_title(title, fontsize=10)
    for row, idx in enumerate(order):
        path  = items[idx]['path']
        mask  = _resize_mask_eval(items[idx]['mask'])
        score = ens_scores[idx]
        img   = np.array(Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
        axes[row, 0].imshow(img)
        axes[row, 1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axes[row, 2].imshow(score, cmap='hot', vmin=0, vmax=score.max()+1e-8)
        axes[row, 3].imshow(img)
        axes[row, 3].imshow(score, cmap='hot', alpha=0.55, vmin=0, vmax=score.max()+1e-8)
        for ax in axes[row]: ax.axis('off')
    fig.suptitle(f'Top anomaly heatmaps — {class_name}', y=1.01)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def _evaluate_class(model, data_root, class_name, memory_banks,
                    seg_heads, val_items, plot_dir, n_vis=4,
                    max_good=150, seed=0):
    W_SH       = 1.0 - W_MB
    good_paths = _load_good_paths(data_root, class_name, max_good, seed)
    if not val_items or not good_paths:
        return {}
    ano_paths = [it['path'] for it in val_items]
    all_paths = good_paths + ano_paths
    n_good    = len(good_paths)

    sh_scores, _ = seghead_infer(model, all_paths, seg_heads[class_name])
    if W_MB > 0:
        mb_scores, _ = memorybank_infer(model, all_paths, memory_banks[class_name])
        ens_scores   = W_MB * mb_scores + W_SH * sh_scores
    else:
        mb_scores  = None
        ens_scores = sh_scores.copy()
    if BLUR_SIGMA > 0:
        ens_scores = np.stack([gaussian_filter(s, sigma=BLUR_SIGMA) for s in ens_scores])

    zero    = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    gt_masks = [zero]*n_good + [_resize_mask_eval(it['mask']) for it in val_items]
    gt_flat  = np.stack(gt_masks).flatten().astype(int)
    if gt_flat.min() == gt_flat.max():
        return {}

    sh_auc  = float(roc_auc_score(gt_flat, sh_scores.flatten()))
    ens_auc = float(roc_auc_score(gt_flat, ens_scores.flatten()))

    image_scores = ens_scores.reshape(len(all_paths), -1).max(axis=1)
    image_labels = [0]*n_good + [1]*len(val_items)
    image_auroc  = float(roc_auc_score(image_labels, image_scores))
    avg_prec     = float(average_precision_score(image_labels, image_scores))

    _save_heatmaps(val_items, ens_scores[n_good:], class_name,
                   plot_dir / f'{class_name}_heatmaps.png', n=n_vis)

    metrics = {
        'pixel_auroc_sh': sh_auc,
        'pixel_auroc_ens': ens_auc,
        'image_auroc_ens': image_auroc,
        'avg_precision_ens': avg_prec,
        'n_good': n_good,
        'n_anomaly': len(val_items),
    }
    if mb_scores is not None:
        metrics['pixel_auroc_mb'] = float(roc_auc_score(gt_flat, mb_scores.flatten()))
    return metrics


def evaluate_all(model, data_root, memory_banks, seg_heads,
                 val_sources, run_dir, n_vis=4, max_good=150, seed=0):
    """Valuta tutte le classi su anomalie di validazione held-out."""
    plot_dir = run_dir / 'evaluation' / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    all_metrics = {}
    print('\nValutazione su anomalie held-out...')
    for class_name in tqdm(sorted(seg_heads.keys()), desc='Classi'):
        m = _evaluate_class(
            model, data_root, class_name, memory_banks, seg_heads,
            val_sources.get(class_name, []), plot_dir,
            n_vis=n_vis, max_good=max_good, seed=seed,
        )
        if m:
            all_metrics[class_name] = m

    use_mb = W_MB > 0
    print(f"\n{'Classe':<12} {'Px-AUC SH':>10} {'Px-AUC Ens':>11} {'Img-AUC':>8} {'AvgPrec':>8}")
    print('-' * 55)
    for cls, m in all_metrics.items():
        print(f"{cls:<12} {m['pixel_auroc_sh']:>10.3f} {m['pixel_auroc_ens']:>11.3f} "
              f"{m['image_auroc_ens']:>8.3f} {m['avg_precision_ens']:>8.3f}")

    if all_metrics:
        keys  = ['pixel_auroc_sh', 'pixel_auroc_ens', 'image_auroc_ens', 'avg_precision_ens']
        means = {k: float(np.mean([m[k] for m in all_metrics.values() if k in m])) for k in keys}
        print('-' * 55)
        print(f"{'MEAN':<12} {means['pixel_auroc_sh']:>10.3f} {means['pixel_auroc_ens']:>11.3f} "
              f"{means['image_auroc_ens']:>8.3f} {means['avg_precision_ens']:>8.3f}")
        all_metrics['_mean'] = means

    metrics_path = run_dir / 'evaluation' / 'metrics.json'
    metrics_path.parent.mkdir(exist_ok=True)
    with open(metrics_path, 'w') as f:
        json.dump(all_metrics, f, indent=2)
    print(f'\nMetriche salvate in {metrics_path}')
    return all_metrics


print("Valutazione definita ✓")

### 4.10 Submission (`src/submission.py`)

In [ ]:
import pandas as pd


def float_matrix_to_q8rle(x: np.ndarray) -> str:
    """Codifica una matrice [0,1] come q8rle column-major."""
    q    = np.clip(np.rint(np.asarray(x, dtype=np.float32) * 255), 0, 255).astype(np.uint8)
    h, w = q.shape
    flat = q.T.reshape(-1)
    if flat.size == 0:
        return f'q8rle {h} {w}'
    cuts   = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    starts = np.r_[0, cuts]
    ends   = np.r_[cuts, flat.size]
    parts  = ['q8rle', str(h), str(w)]
    for v, n in zip(flat[starts], ends - starts):
        parts += [str(int(v)), str(int(n))]
    return ' '.join(parts)


def encode_submission(mb_test, sh_test, mb_train, sh_train, run_dir):
    """Normalizza, ensemble e codifica le predizioni in un CSV di submission."""
    W_SH = 1.0 - W_MB
    print('\nEncoding submission...')
    rows = []
    for class_name in sorted(sh_test.keys()):
        sh_lo, sh_hi = np.percentile(sh_train[class_name].flatten(), [P_LO, P_HI])
        for fn in sorted(sh_test[class_name].keys()):
            n_sh = np.clip((sh_test[class_name][fn] - sh_lo) / (sh_hi - sh_lo + 1e-8), 0, 1)
            ens  = W_SH * n_sh
            if W_MB > 0:
                mb_lo, mb_hi = np.percentile(mb_train[class_name].flatten(), [P_LO, P_HI])
                n_mb = np.clip((mb_test[class_name][fn] - mb_lo) / (mb_hi - mb_lo + 1e-8), 0, 1)
                ens  = ens + W_MB * n_mb
            if BLUR_SIGMA > 0:
                ens = gaussian_filter(ens, sigma=BLUR_SIGMA)
            ens = np.clip(ens, 0, 1).astype(np.float32)
            rows.append({'ID': fn[:-4], 'Label': float_matrix_to_q8rle(ens)})

    ts       = run_dir.name
    csv_path = run_dir / f'submission_{ts}.csv'
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f'Submission salvata → {csv_path}')
    return csv_path


print("Submission definita ✓")

## 5. Training

Carica il backbone DINOv2 e avvia il training completo.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='xFormers is not available')

# ── Parametri run ─────────────────────────────────────────────────────────────
RUN_SEED          = SEED
RUN_CORESET_RATIO = CORESET_RATIO
RUN_CORESET_METHOD = 'random'   # 'random' | 'greedy'
RUN_EPOCHS        = EPOCHS
RUN_PATIENCE      = PATIENCE
N_VIS             = 4
EVAL_MAX_GOOD     = 150

set_all_seeds(RUN_SEED)
print(f'Seed: {RUN_SEED} | Device: {device} | '
      f'Coreset ratio: {RUN_CORESET_RATIO*100:.1f}% | '
      f'Coreset method: {RUN_CORESET_METHOD}')

In [ ]:
print('Caricamento DINOv2...')
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
dinov2 = dinov2.to(device).eval()
for p in dinov2.parameters():
    p.requires_grad = False
print('DINOv2 caricato ✓')

In [ ]:
if W_MB > 0:
    memory_banks = build_memory_banks(
        dinov2, DATA_ROOT,
        coreset_ratio=RUN_CORESET_RATIO,
        seed=RUN_SEED,
        coreset_method=RUN_CORESET_METHOD,
    )
else:
    memory_banks = {}
    print('W_MB=0: memory bank construction skipped')

In [ ]:
train_sources, val_sources = collect_anomaly_sources(DATA_ROOT)
seg_heads = train_seg_heads(
    dinov2, DATA_ROOT, train_sources, val_sources,
    epochs=RUN_EPOCHS, patience=RUN_PATIENCE, seed=RUN_SEED,
)
torch.cuda.empty_cache()
gc.collect()
print('Training completato ✓')

## 6. Valutazione (proxy metrics su dati di training)

In [ ]:
import shutil

eval_tmp      = OUTPUT_DIR / '_eval_tmp'
eval_max_good = EVAL_MAX_GOOD if EVAL_MAX_GOOD > 0 else None

metrics = evaluate_all(
    dinov2, DATA_ROOT, memory_banks, seg_heads, val_sources,
    run_dir=eval_tmp,
    n_vis=N_VIS,
    max_good=eval_max_good,
    seed=RUN_SEED,
)
torch.cuda.empty_cache()

In [ ]:
# Visualizza le heatmap nel notebook
from IPython.display import Image as IPImage, display

plot_dir = eval_tmp / 'evaluation' / 'plots'
heatmap_files = sorted(plot_dir.glob('*_heatmaps.png')) if plot_dir.exists() else []
for f in heatmap_files:
    print(f'\n--- {f.stem} ---')
    display(IPImage(filename=str(f)))

## 7. Inferenza & Submission

In [ ]:
mb_train, mb_test, sh_train, sh_test = run_inference(
    dinov2, DATA_ROOT, memory_banks, seg_heads,
)
torch.cuda.empty_cache()

In [ ]:
config_dict = {
    'seed': RUN_SEED,
    'coreset_ratio': RUN_CORESET_RATIO,
    'epochs': RUN_EPOCHS,
    'data_root': str(DATA_ROOT),
}
run_dir = save_run(
    OUTPUT_DIR, memory_banks, seg_heads,
    config_dict=config_dict, metrics=metrics,
)
if eval_tmp.exists():
    shutil.copytree(eval_tmp, run_dir / 'evaluation', dirs_exist_ok=True)
    shutil.rmtree(eval_tmp)

csv_path = encode_submission(mb_test, sh_test, mb_train, sh_train, run_dir)
print(f'\nTutti gli output in: {run_dir}')
print(f'Submission CSV:       {csv_path}')

---
## 📝 TODO / Prossimi esperimenti

- [ ] Verificare anomaly maps senza contributo del patch core
- [ ] Separare il background dagli oggetti prima di calcolare l'anomalia (rumore di fondo)
- [ ] Greedy subsampling per la memory bank
- [ ] Grid search sugli iperparametri
- [ ] Bilanciare i contributi dell'ensemble: SegHead è più localizzato (meglio per AP), Memory Bank produce score più diffusi
- [ ] Percentuale di campioni puliti nel training del dataset sintetico
- [ ] Normalizzazione per-classe: provare a usare score misti train+test per il percentile superiore
- [ ] Test-time augmentation (h-flip già implementato, aggiungere v-flip)
- [ ] Visualizzazioni più ricche di train e test statistics